<a href="https://colab.research.google.com/github/sulucay01/multimodal-rs-segmentation/blob/dev/notebooks/06_qualitative_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 — Qualitative analysis

DI725 Term Project — Multimodal Fusion for Remote Sensing Land Cover Segmentation

This notebook is a visual and analytical complement to the experimental notebooks (`01_data_exploration.ipynb` through `05_phase3_ablations.ipynb`). It addresses two questions that the aggregated mIoU tables cannot answer on their own:

1. **What does the input-output behavior of the model look like in practice?** Shown via curated test-set examples comparing predictions across the vision-only baseline, Phase 2 multimodal, and Phase 3 final model.

2. **Where does the model fail, and what does the failure structure look like?** Shown via a confusion matrix and a set of worst-case failure scenes.

The notebook is purely analytical — no models are trained here. Predictions are computed once via test-set inference on already-trained checkpoints, cached to disk, then loaded as needed by the visualization cells.

## 1. Setup

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install dependencies (same versions as the model-training notebooks)
!pip install transformers timm einops open_clip_torch huggingface_hub -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.6 MB/s eta 0:00:00


In [ ]:
# Project paths and a local working copy for fast I/O
from pathlib import Path
import shutil

PROJECT_DIR      = Path('/content/drive/MyDrive/DI725_Project')
DATA_DIR_DRIVE   = PROJECT_DIR / 'data'
SPLITS_CSV       = DATA_DIR_DRIVE / 'captions_with_splits.csv'
CHECKPOINTS_DIR  = PROJECT_DIR / 'checkpoints'
RESULTS_DIR      = PROJECT_DIR / 'results'
PREDICTIONS_DIR  = RESULTS_DIR / 'predictions'
REMOTECLIP_CACHE = PROJECT_DIR / 'remoteclip_cache'

PREDICTIONS_DIR.mkdir(exist_ok=True)

# Local SSD copy avoids Drive I/O bottleneck during inference
LOCAL_DATA        = Path('/content/data')
LOCAL_IMAGES      = LOCAL_DATA / 'images'
LOCAL_MASKS_CLASS = LOCAL_DATA / 'masks_class'

if not LOCAL_DATA.exists():
    print('Copying data to local SSD...')
    shutil.copytree(DATA_DIR_DRIVE / 'images', LOCAL_IMAGES)
    shutil.copytree(DATA_DIR_DRIVE / 'masks_class', LOCAL_MASKS_CLASS)
    print('Done.')
else:
    print('Local data already present.')

Copying data to local SSD...


In [ ]:
# Class definitions (same across all project notebooks)
CLASS_NAMES = ['Tree', 'Shrub', 'Grass', 'Crop', 'Built-up', 'Barren', 'Water']
NUM_CLASSES = len(CLASS_NAMES)

# Canonical RGB palette (used to colorize masks and predictions)
CLASS_RGB = {
    'Tree':     (  0, 100,   0),
    'Shrub':    (255, 182, 193),
    'Grass':    (154, 205,  50),
    'Crop':     (255, 215,   0),
    'Built-up': (139,  69,  19),
    'Barren':   (211, 211, 211),
    'Water':    (  0,   0, 255),
}

# Lookup tensor for fast mask-to-RGB conversion via index lookup
CLASS_PALETTE = np.array([CLASS_RGB[name] for name in CLASS_NAMES], dtype=np.uint8)

# The caption variant used by Phase 2 multimodal and Phase 3 final model
BEST_CAPTION = 'text_qwen3-4b'

# Load splits
split_df = pd.read_csv(SPLITS_CSV)
test_df  = split_df[split_df['split'] == 'test'].reset_index(drop=True)
train_df = split_df[split_df['split'] == 'train'].reset_index(drop=True)
print(f'Train: {len(train_df)} samples, Test: {len(test_df)} samples')

In [ ]:
# Class definitions (same across all project notebooks)
CLASS_NAMES = ['Tree', 'Shrub', 'Grass', 'Crop', 'Built-up', 'Barren', 'Water']
NUM_CLASSES = len(CLASS_NAMES)

# Canonical RGB palette (used to colorize masks and predictions)
CLASS_RGB = {
    'Tree':     (  0, 100,   0),
    'Shrub':    (255, 182, 193),
    'Grass':    (154, 205,  50),
    'Crop':     (255, 215,   0),
    'Built-up': (139,  69,  19),
    'Barren':   (211, 211, 211),
    'Water':    (  0,   0, 255),
}

# Lookup tensor for fast mask-to-RGB conversion via index lookup
CLASS_PALETTE = np.array([CLASS_RGB[name] for name in CLASS_NAMES], dtype=np.uint8)

# The caption variant used by Phase 2 multimodal and Phase 3 final model
BEST_CAPTION = 'text_qwen3-4b'

# Load splits
split_df = pd.read_csv(SPLITS_CSV)
test_df  = split_df[split_df['split'] == 'test'].reset_index(drop=True)
train_df = split_df[split_df['split'] == 'train'].reset_index(drop=True)
print(f'Train: {len(train_df)} samples, Test: {len(test_df)} samples')